# HW4 : Tweet Sentiment Classification Challenge

### Implement model that predicts sentiment of a tweet based on its text content.

## Installations & Load
 Install the necessary libraries, and load datasets

In [57]:
!pip install -q transformers datasets accelerate scikit-learn pandas torch kaggle

import libraries


In [58]:
import re
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
  AutoTokenizer,
  AutoModelForSequenceClassification,
  Trainer,
  TrainingArguments,
  DataCollatorWithPadding
)

Load Datasets

In [59]:
# Create kaggle directory
!mkdir -p ~/.kaggle

In [60]:
# Import API key to google colab
from google.colab import files
files.upload()

# Move token file
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

Saving kaggle.json to kaggle (2).json


In [61]:
# Download Competition Dataset
!kaggle competitions download -c twitter-sentiment-classification-challenge


twitter-sentiment-classification-challenge.zip: Skipping, found more recently modified local copy (use --force to force download)


In [62]:
# Unzip Datasets
!unzip twitter-sentiment-classification-challenge.zip -d data/

Archive:  twitter-sentiment-classification-challenge.zip
replace data/HW4_test_no-label.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: data/HW4_test_no-label.csv  
replace data/HW4_train.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: data/HW4_train.csv      


In [63]:
# Load Datasets
train_path = "/content/data/HW4_train.csv"
test_path = "/content/data/HW4_test_no-label.csv"

train_df = pd.read_csv(train_path, encoding='latin1')
test_df = pd.read_csv(test_path, encoding='latin1')

## Implementing & Train
 Implement Model, and train model

In [107]:
# Sentiment Label list
label_list = [
  "Positive",
  "Negative",
  "Neutral",
  "Extremely_Positive",
  "Extremely_Negative",
]

label2id = {label: idx for idx, label in enumerate(label_list)}
id2label = {idx: label for label, idx in label2id.items()}

train_df["label"] = train_df["Sentiment"].map(label2id)

In [134]:
# Preprocess tweets

def clean_tweet(text):
    text = text.lower()
    text = re.sub(r"http\\S+", "", text)
    text = re.sub(r"@\\w+", "", text)
    text = re.sub(r"\\s+", " ", text).strip()
    return text

train_df["OriginalTweet"] = train_df["OriginalTweet"].astype(str).apply(clean_tweet)
test_df["OriginalTweet"] = test_df["OriginalTweet"].astype(str).apply(clean_tweet)

In [135]:
# Train / Validation Split
train_df_cleaned = train_df.dropna(subset=['label'])

train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df_cleaned["OriginalTweet"].tolist(),
    train_df_cleaned["label"].tolist(),
    test_size=0.1,
    random_state=42,
    stratify=train_df_cleaned["label"]
)

In [136]:
# Tokenizer & Dataset

model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)


def tokenize(texts):
  return tokenizer(texts, truncation=True)


class TextDataset(torch.utils.data.Dataset):
  def __init__(self, texts, labels=None):
    self.encodings = tokenizer(texts, truncation=True, padding=False)
    self.labels = labels


  def __getitem__(self, idx):
    item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
    if self.labels is not None:
      item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
    return item


  def __len__(self):
    return len(self.encodings["input_ids"])


train_dataset = TextDataset(train_texts, train_labels)
val_dataset = TextDataset(val_texts, val_labels)


data_collator = DataCollatorWithPadding(tokenizer)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [137]:
# Balance class weights

# Get unique labels actually present in the cleaned training data
present_labels = np.unique(train_df_cleaned["label"]).astype(int)

# Calculate weights only for the present labels
calculated_weights = compute_class_weight(
    class_weight="balanced",
    classes=present_labels,
    y=train_df_cleaned["label"]
)

# Create a full weight tensor for all 5 classes, default to 1.0
full_class_weights = np.ones(len(label_list), dtype=np.float32)

# Assign calculated weights to the corresponding present labels
for idx, label_val in enumerate(present_labels):
    full_class_weights[label_val] = calculated_weights[idx]

class_weights = torch.tensor(full_class_weights, dtype=torch.float)

In [138]:
# Set Model
model = AutoModelForSequenceClassification.from_pretrained(
  model_name,
  num_labels=len(label_list),
  id2label=id2label,
  label2id=label2id,
  problem_type="single_label_classification"
)

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [139]:
class WeightedTrainer(Trainer):
  def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=0):
    labels = inputs.get("labels")
    outputs = model(**inputs)
    logits = outputs.get("logits")
    loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights.to(logits.device))
    loss = loss_fct(logits, labels)
    return (loss, outputs) if return_outputs else loss

In [140]:
# Define Metrics

def compute_metrics(eval_pred):
  logits, labels = eval_pred
  preds = np.argmax(logits, axis=1)
  return {
    "accuracy": accuracy_score(labels, preds),
    "f1": f1_score(labels, preds, average="macro")
}

In [141]:
# Set Training Arguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    warmup_ratio=0.1,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True,
    logging_steps=100,
    report_to="none"
)

In [142]:
# Train model

trainer = WeightedTrainer(
  model=model,
  args=training_args,
  train_dataset=train_dataset,
  eval_dataset=val_dataset,
  tokenizer=tokenizer,
  data_collator=data_collator,
  compute_metrics=compute_metrics
)

trainer.train()

/tmp/ipython-input-2743594067.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedTrainer.__init__`. Use `processing_class` instead.
  trainer = WeightedTrainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.951100,0.860205,0.617343,0.615286
2,0.839900,0.812646,0.656573,0.653651
3,0.763900,0.786798,0.668961,0.668458


TrainOutput(global_step=2454, training_loss=0.9047857314081045, metrics={'train_runtime': 619.6628, 'train_samples_per_second': 126.582, 'train_steps_per_second': 3.96, 'total_flos': 3111346339505700.0, 'train_loss': 0.9047857314081045, 'epoch': 3.0})

In [143]:
# Inference on Test Set

test_dataset = TextDataset(test_df["OriginalTweet"].tolist())
preds = trainer.predict(test_dataset)


test_pred_ids = np.argmax(preds.predictions, axis=1)


test_df["Sentiment"] = [id2label[i] for i in test_pred_ids]

In [144]:
# Save Output as Inference.csv

inference_df = test_df[["UserName", "Sentiment"]]

output_path = "/content/inference.csv"
inference_df.to_csv(output_path, index=False)

print("Saved:", output_path)

Saved: /content/inference.csv
